<a href="https://colab.research.google.com/github/AlexJoaquimPereira/FortiPrompt-redteam/blob/feature%2FPretrainLM/FortiPrompt_RedTeam_PretrainLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

# REQUIRED for WGAN-GP + Transformers
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
!pip install torch transformers sentence-transformers pandas tqdm


In [ ]:
import pandas as pd

df = pd.read_csv("/content/malicious_prompts_dataset_combined.csv")
real_prompts = df["prompt"].astype(str).tolist()

print("Loaded prompts:", len(real_prompts))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

generator = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
generator.train()

for name, param in generator.named_parameters():
    if "transformer.h.5" not in name:
        param.requires_grad = False


In [ ]:
import torch.nn as nn

class Critic(nn.Module):
    def __init__(self, vocab_size, d_model=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=4,
                batch_first=True
            ),
            num_layers=3
        )
        self.fc = nn.Linear(d_model, 1)

    def forward(self, ids, mask=None):
        emb = self.embed(ids)
        if mask is not None:
            mask = ~mask.bool()
        enc = self.encoder(emb, src_key_padding_mask=mask)
        return self.fc(enc.mean(dim=1))

    def forward_embeddings(self, emb, mask=None):
        if mask is not None:
            mask = ~mask.bool()
        enc = self.encoder(emb, src_key_padding_mask=mask)
        return self.fc(enc.mean(dim=1))


critic = Critic(tokenizer.vocab_size).to(device)


In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer("all-MiniLM-L6-v2").to(device)

def semantic_similarity(fake_text, real_batch):
    f = sbert.encode(fake_text, convert_to_tensor=True)
    r = sbert.encode(real_batch, convert_to_tensor=True)
    return torch.cosine_similarity(f.unsqueeze(0), r).max()


In [ ]:
def gradient_penalty(critic, real_ids, fake_ids, real_mask):
    real_emb = critic.embed(real_ids)
    fake_emb = critic.embed(fake_ids)

    alpha = torch.rand(real_emb.size(0), 1, 1, device=device)
    interp = alpha * real_emb + (1 - alpha) * fake_emb
    interp.requires_grad_(True)

    score = critic.forward_embeddings(interp, real_mask)

    grads = torch.autograd.grad(
        outputs=score,
        inputs=interp,
        grad_outputs=torch.ones_like(score),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    return ((grads.norm(2, dim=2) - 1) ** 2).mean()


In [ ]:
opt_G = torch.optim.Adam(
    filter(lambda p: p.requires_grad, generator.parameters()),
    lr=1e-5
)
opt_C = torch.optim.Adam(critic.parameters(), lr=1e-4)

LAMBDA_GP = 10
SEM_THRESHOLD = 0.65


In [ ]:
from random import sample
from tqdm import trange

MAX_LEN = 64
BATCH = 8

for step in trange(2000):

    # ---- Critic ----
    for _ in range(5):
        real_texts = sample(real_prompts, BATCH)
        enc_real = tokenizer(
            real_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN
        )

        real_ids = enc_real.input_ids.to(device)
        real_mask = enc_real.attention_mask.to(device)

        # Prepare input_ids for batch generation
        input_ids_for_gen = torch.tensor([[tokenizer.bos_token_id]] * BATCH).to(device)

        # Generate BATCH fake IDs (they will have varying lengths unless MAX_LEN is reached)
        fake_ids_generated = generator.generate(
            input_ids=input_ids_for_gen,
            max_length=MAX_LEN,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=1 # This means one sequence per row in input_ids
        )

        # Decode to text and then re-tokenize with padding to ensure consistent MAX_LEN
        fake_texts_for_critic = [tokenizer.decode(ids, skip_special_tokens=True) for ids in fake_ids_generated]

        enc_fake = tokenizer(
            fake_texts_for_critic,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN
        )
        fake_ids = enc_fake.input_ids.to(device)
        fake_mask = enc_fake.attention_mask.to(device)

        loss_c = -(critic(real_ids, real_mask).mean()
                   - critic(fake_ids, fake_mask).mean())

        gp = gradient_penalty(
            critic,
            real_ids,
            fake_ids,
            real_mask
        )

        (loss_c + LAMBDA_GP * gp).backward()
        opt_C.step()
        opt_C.zero_grad()

    # ---- Generator ----
    fake_ids = generator.generate(
        input_ids=torch.tensor([[tokenizer.bos_token_id]]).to(device),
        max_length=MAX_LEN,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    fake_text = tokenizer.decode(fake_ids[0], skip_special_tokens=True)
    sem = semantic_similarity(fake_text, real_prompts[:50])

    adv_loss = -critic(fake_ids).mean()

    if sem < SEM_THRESHOLD:
        adv_loss += 1.0  # semantic penalty

    adv_loss.backward()
    opt_G.step()
    opt_G.zero_grad()

    if step % 100 == 0:
        print(f"Step {step} | Adv {adv_loss.item():.3f} | Sem {sem:.2f}")

In [ ]:
generator.eval()

for _ in range(10):
    out = generator.generate(
        input_ids=torch.tensor([[tokenizer.bos_token_id]]).to(device),
        max_length=64,
        do_sample=True,
        top_p=0.9
    )
    print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
SAVE_DIR = "/content/saved_generator"

generator.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

In [ ]:
torch.save(
    critic.state_dict(),
    "/content/critic_wgan_gp.pt"
)


In [ ]:
import json

metadata = {
    "model": "distilgpt2",
    "training_steps": 2000,
    "wgan_gp": True,
    "semantic_threshold": SEM_THRESHOLD,
    "max_length": 64,
    "notes": "Final FYP checkpoint"
}

with open("/content/training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


In [ ]:
from google.colab import files

files.download("/content/critic_wgan_gp.pt")
files.download("/content/training_metadata.json")

# Zip generator folder
!zip -r saved_generator.zip /content/saved_generator
files.download("saved_generator.zip")
